# Day 1 — Dataset Collection
**Goal:** Collect 1,500 Indian legal instruction pairs from IndianKanoon + GPT-4o synthetic generation.

**Outputs:**
- `output/raw_judgments.json` — scraped judgment texts
- `output/dataset_alpaca.json` — Alpaca format (human review)
- `output/train.jsonl` — 1,350 training samples (chat format)
- `output/eval.jsonl` — 150 evaluation samples (chat format)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# ── 0. Setup ──────────────────────────────────────────────────────────────────
import sys, os
from pathlib import Path
from dotenv import load_dotenv

# Add project root so imports work from notebooks/
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.chdir(PROJECT_ROOT)   # ensure relative paths resolve correctly

# Load Azure credentials (and any other secrets) from .env
load_dotenv(PROJECT_ROOT / '.env', override=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Python: {sys.version}')

Project root: /home/ubuntu/Fine-tuning/NyayaGPT
Python: 3.10.13 (main, Mar 19 2026, 17:06:20) [GCC 9.4.0]


In [3]:
# ── 1. Check dependencies ────────────────────────────────────────────────────
import subprocess, sys

INSTALL = [
    'python-dotenv', # .env loading
    'requests', 'beautifulsoup4', 'lxml',
    'openai',        # Azure OpenAI
    'rouge-score',   # quality evaluation
]

for pkg in INSTALL:
    import_name = pkg.replace('-', '_')
    try:
        __import__(import_name)
        print(f'  ✓ {pkg}')
    except ImportError:
        print(f'  Installing {pkg} …')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

  Installing python-dotenv …



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


  ✓ requests
  Installing beautifulsoup4 …



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


  ✓ lxml
  ✓ openai
  ✓ rouge-score


In [4]:
# ── 2. Azure OpenAI check (optional) ─────────────────────────────────────────
import os

AZURE_KEY      = os.getenv('AZURE_OPENAI_KEY', '')
AZURE_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT', '')

if AZURE_KEY:
    print('✓ Azure OpenAI configured — will use GPT-4o for generation')
else:
    print('⚠ AZURE_OPENAI_KEY not set — will fall back to local GGUF teacher')
    print('  Set it with: export AZURE_OPENAI_KEY=your_key')
    print('               export AZURE_OPENAI_ENDPOINT=https://your-endpoint.openai.azure.com')

✓ Azure OpenAI configured — will use GPT-4o for generation


## Section 1 — IndianKanoon Scraper
Fetches judgment texts across 15 legal topic queries.

In [5]:
from src.nyaya_pipeline.data_collector import collect_judgments, DEFAULT_QUERIES

print(f'Queries: {len(DEFAULT_QUERIES)}')
for q in DEFAULT_QUERIES:
    print(f'  • {q}')

Queries: 15
  • IPC section 302 murder
  • IPC section 420 cheating fraud
  • criminal appeal High Court
  • constitutional rights fundamental article 21
  • PIL public interest litigation Supreme Court
  • property dispute civil suit
  • cyber crime Information Technology Act
  • contract law breach damages
  • domestic violence Protection of Women Act
  • land acquisition compensation
  • copyright infringement intellectual property India
  • motor accident compensation MACT
  • cheque dishonour NI Act section 138
  • bail application Sessions Court
  • habeas corpus writ petition


In [6]:
# Run scraper
# Adjust max_docs for a quick test (e.g., 20) or full run (600)
judgments = collect_judgments(
    queries=DEFAULT_QUERIES,
    pages_per_query=2,
    max_docs=600,          # <- set to 20 for a quick test
    rate_limit_secs=1.5,   # be polite to IndianKanoon
)

total_chunks = sum(len(j['chunks']) for j in judgments)
print(f'\n✓ Collected {len(judgments)} judgments with {total_chunks} chunks')

2026-04-18 16:04:05,060 [INFO] src.nyaya_pipeline.data_collector: Scraping query: 'IPC section 302 murder'
2026-04-18 16:04:45,542 [INFO] src.nyaya_pipeline.data_collector: Scraping query: 'IPC section 420 cheating fraud'
2026-04-18 16:05:26,324 [INFO] src.nyaya_pipeline.data_collector: Scraping query: 'criminal appeal High Court'
2026-04-18 16:06:07,789 [INFO] src.nyaya_pipeline.data_collector: Scraping query: 'constitutional rights fundamental article 21'
2026-04-18 16:06:49,561 [INFO] src.nyaya_pipeline.data_collector: Scraping query: 'PIL public interest litigation Supreme Court'
2026-04-18 16:07:32,670 [INFO] src.nyaya_pipeline.data_collector: Scraping query: 'property dispute civil suit'
2026-04-18 16:08:12,986 [INFO] src.nyaya_pipeline.data_collector: Scraping query: 'cyber crime Information Technology Act'
2026-04-18 16:08:48,524 [INFO] src.nyaya_pipeline.data_collector: Scraping query: 'contract law breach damages'
2026-04-18 16:09:27,522 [INFO] src.nyaya_pipeline.data_collect

In [8]:
import json
judgments=json.loads(Path("/home/ubuntu/Fine-tuning/NyayaGPT/output/raw_judgments.json").read_text())

In [9]:
# Inspect a sample
import json
sample = judgments[0] if judgments else {}
print(json.dumps({k: v if k != 'chunks' else f'[{len(v)} chunks]'
                  for k, v in sample.items()}, indent=2))
if sample.get('chunks'):
    print('\nFirst chunk:')
    print(sample['chunks'][0][:400])

{
  "title": "Surender Alias Choti vs State Of Haryana And Others on 20 October, 2022",
  "url": "https://indiankanoon.org/doc/103473592/",
  "year": "2022",
  "text": "[Cites 34 , Cited by 2 ] Punjab-Haryana High Court Surender Alias Choti vs State Of Haryana And Others on 20 October, 2022 Author: Tejinder Singh Dhindsa Bench: Tejinder Singh Dhindsa CRWP-1324-2022 1 IN THE HIGH COURT OF PUNJAB AND HARYANA AT CHANDIGARH CRWP-1324-2022 Date of Decision: 20.10.2022 SURENDER @ CHOTI ...Petitioner V/S STATE OF HARYANA AND OTHERS ...Respondents CORAM: HON'BLE MR. JUSTICE TEJINDER SINGH DHINDSA HON'BLE MR. JUSTICE DEEPAK MANCHANDA Present: Ms. Amrita Garg, Advocate for the petitioner. Mr. Randhir Singh, Addl. AG Haryana. DEEPAK MANCHANDA J. The instant petition has been filed challenging the impugned order dated 28.01.2022 (Annexure P-13) passed by respondent No. 6 vide which the application for grant of furlough to the petitioner had been declined on the ground that the petitioner falls int

## Section 2 — Synthetic Q&A Generation
Uses Azure OpenAI GPT-4o if key is set, otherwise falls back to the local GGUF teacher.

In [10]:
from src.nyaya_pipeline.synthetic_generator import generate_qa_pairs, save_datasets

# Flatten all chunks from collected judgments
all_chunks = [chunk for j in judgments for chunk in j['chunks']]
print(f'Total chunks available: {len(all_chunks)}')
print(f'Target pairs: {len(all_chunks) * 3} (3 per chunk)')

Total chunks available: 30661
Target pairs: 91983 (3 per chunk)


In [11]:
import torch
torch.cuda.is_available(), torch.cuda.device_count(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A' 

(True, 1, 'NVIDIA GeForce RTX 5090')

In [6]:
import json
with open('output/raw_judgments.json') as f:
    judgments = json.load(f)

SAFE_QUERIES = {
    'constitutional rights fundamental article 21',
    'PIL public interest litigation Supreme Court',
    'property dispute civil suit',
    'contract law breach damages',
    'land acquisition compensation',
    'copyright infringement intellectual property India',
    'motor accident compensation MACT',
    'cheque dishonour NI Act section 138',
    'habeas corpus writ petition',
}

azure_chunks = [c for j in judgments if j['query'] in SAFE_QUERIES for c in j['chunks']][:1]
gguf_chunks  = [c for j in judgments for c in j['chunks']][:2]   # any topic

print(f'Azure: {len(azure_chunks)} chunks → ~300 pairs (~15 min, ~$1.50)')
print(f'GGUF:  {len(gguf_chunks)} chunks → ~1200 pairs (~13 min)')


Azure: 1 chunks → ~300 pairs (~15 min, ~$1.50)
GGUF:  2 chunks → ~1200 pairs (~13 min)


In [9]:
from src.nyaya_pipeline.synthetic_generator import generate_qa_pairs, save_datasets
# 1. Azure run — ensure key is loaded
from dotenv import load_dotenv; load_dotenv('.env', override=True)
alpaca_a, chat_a = generate_qa_pairs(azure_chunks, qa_per_chunk=3, include_cot_every_n=1)

# 2. Switch to GGUF — blank the key
import os; os.environ['AZURE_OPENAI_KEY'] = ''
# also reset module cache if needed
alpaca_g, chat_g = generate_qa_pairs(gguf_chunks, qa_per_chunk=3, include_cot_every_n=1)

# 3. Merge + save
alpaca_all = alpaca_a + alpaca_g
chat_all   = chat_a   + chat_g
save_datasets(alpaca_all, chat_all)


2026-04-19 01:27:33,161 [INFO] src.nyaya_pipeline.synthetic_generator: Using Azure OpenAI GPT-4o for generation
2026-04-19 01:27:45,331 [INFO] src.nyaya_pipeline.synthetic_generator: Generation complete: 2 accepted, 2 rejected (50.0% acceptance)
2026-04-19 01:27:45,332 [INFO] src.nyaya_pipeline.synthetic_generator: Using Azure OpenAI GPT-4o for generation
2026-04-19 01:28:03,991 [INFO] src.nyaya_pipeline.synthetic_generator: Generation complete: 8 accepted, 0 rejected (100.0% acceptance)
2026-04-19 01:28:03,993 [INFO] src.nyaya_pipeline.synthetic_generator: Saved 9 train / 1 eval samples
2026-04-19 01:28:03,994 [INFO] src.nyaya_pipeline.synthetic_generator:   train → /home/ubuntu/Fine-tuning/NyayaGPT/output/train.jsonl
2026-04-19 01:28:03,994 [INFO] src.nyaya_pipeline.synthetic_generator:   eval  → /home/ubuntu/Fine-tuning/NyayaGPT/output/eval.jsonl
2026-04-19 01:28:03,995 [INFO] src.nyaya_pipeline.synthetic_generator:   alpaca→ /home/ubuntu/Fine-tuning/NyayaGPT/output/dataset_alpaca.j

(PosixPath('/home/ubuntu/Fine-tuning/NyayaGPT/output/train.jsonl'),
 PosixPath('/home/ubuntu/Fine-tuning/NyayaGPT/output/eval.jsonl'))

In [ ]:
import json, os
from dotenv import load_dotenv
from src.nyaya_pipeline import config
from src.nyaya_pipeline.synthetic_generator import generate_qa_pairs, save_datasets

# ── Build chunk lists ────────────────────────────────────────────────────────
with open('output/raw_judgments.json') as f:
    judgments = json.load(f)

SAFE_QUERIES = {
    'constitutional rights fundamental article 21',
    'PIL public interest litigation Supreme Court',
    'property dispute civil suit',
    'contract law breach damages',
    'land acquisition compensation',
    'copyright infringement intellectual property India',
    'motor accident compensation MACT',
    'cheque dishonour NI Act section 138',
    'habeas corpus writ petition',
}

azure_chunks = [c for j in judgments if j['query'] in SAFE_QUERIES for c in j['chunks']][:150]
gguf_chunks  = [c for j in judgments for c in j['chunks']][:400]
print(f'Azure: {len(azure_chunks)} chunks | GGUF: {len(gguf_chunks)} chunks')

# ── 1. Azure run ─────────────────────────────────────────────────────────────
load_dotenv('.env', override=True)
config.AZURE_OPENAI_KEY      = os.getenv('AZURE_OPENAI_KEY', '')
config.AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT', '')
assert config.AZURE_OPENAI_KEY, 'AZURE_OPENAI_KEY missing in .env'

alpaca_a, chat_a = generate_qa_pairs(azure_chunks, qa_per_chunk=3, include_cot_every_n=1)
print(f'Azure produced {len(alpaca_a)} pairs')

# ── 2. GGUF run (force it by blanking config directly) ───────────────────────
config.AZURE_OPENAI_KEY = ''
alpaca_g, chat_g = generate_qa_pairs(gguf_chunks, qa_per_chunk=3, include_cot_every_n=1)
print(f'GGUF produced {len(alpaca_g)} pairs')

# ── 3. Merge + save ──────────────────────────────────────────────────────────
alpaca_all = alpaca_a + alpaca_g
chat_all   = chat_a   + chat_g
save_datasets(alpaca_all, chat_all)
print(f'TOTAL: {len(alpaca_all)} pairs saved')


Azure: 150 chunks | GGUF: 400 chunks
2026-04-19 01:37:18,177 [INFO] src.nyaya_pipeline.synthetic_generator: Using Azure OpenAI GPT-4o for generation
2026-04-19 01:40:44,606 [INFO] src.nyaya_pipeline.synthetic_generator: Progress: 20 chunks processed → 40 samples collected, 40 rejected
2026-04-19 01:44:41,747 [INFO] src.nyaya_pipeline.synthetic_generator: Progress: 40 chunks processed → 81 samples collected, 79 rejected
2026-04-19 01:48:26,660 [INFO] src.nyaya_pipeline.synthetic_generator: Progress: 60 chunks processed → 118 samples collected, 122 rejected
2026-04-19 01:51:52,072 [INFO] src.nyaya_pipeline.synthetic_generator: Progress: 80 chunks processed → 158 samples collected, 162 rejected
2026-04-19 01:55:23,800 [INFO] src.nyaya_pipeline.synthetic_generator: Progress: 100 chunks processed → 208 samples collected, 192 rejected
2026-04-19 01:58:38,400 [INFO] src.nyaya_pipeline.synthetic_generator: Progress: 120 chunks processed → 288 samples collected, 192 rejected
2026-04-19 02:01:54

In [21]:
# Generate Q&A pairs
# For a quick test, slice to 10 chunks:
all_chunks = all_chunks[:1500]

alpaca_samples, chat_samples = generate_qa_pairs(
    chunks=all_chunks,
    qa_per_chunk=3,
    include_cot_every_n=1,
)

print(f'\n✓ Generated {len(alpaca_samples)} accepted Q&A pairs')

train_path, eval_path = save_datasets(alpaca_samples, chat_samples)

# Verify
def count_lines(p):
    return sum(1 for _ in open(p, encoding='utf-8'))

print(f'\n✓ Dataset saved:')
print(f'  train.jsonl : {count_lines(train_path)} samples')
print(f'  eval.jsonl  : {count_lines(eval_path)} samples')

2026-04-19 01:16:58,513 [INFO] src.nyaya_pipeline.synthetic_generator: Using local GGUF teacher for generation (Azure key not set)
2026-04-19 01:17:36,273 [INFO] src.nyaya_pipeline.synthetic_generator: Generation complete: 57 accepted, 3 rejected (95.0% acceptance)

✓ Generated 57 accepted Q&A pairs
2026-04-19 01:17:36,276 [INFO] src.nyaya_pipeline.synthetic_generator: Saved 51 train / 6 eval samples
2026-04-19 01:17:36,277 [INFO] src.nyaya_pipeline.synthetic_generator:   train → /home/ubuntu/Fine-tuning/NyayaGPT/output/train.jsonl
2026-04-19 01:17:36,277 [INFO] src.nyaya_pipeline.synthetic_generator:   eval  → /home/ubuntu/Fine-tuning/NyayaGPT/output/eval.jsonl
2026-04-19 01:17:36,278 [INFO] src.nyaya_pipeline.synthetic_generator:   alpaca→ /home/ubuntu/Fine-tuning/NyayaGPT/output/dataset_alpaca.json

✓ Dataset saved:
  train.jsonl : 51 samples
  eval.jsonl  : 6 samples


## Section 3 — Review Sample Quality

In [16]:
# Alpaca format (human-readable review)
import random
for i, sample in enumerate(random.sample(alpaca_samples, min(5, len(alpaca_samples)))):
    print(f'--- Sample {i+1} ---')
    print(f'Instruction: {sample["instruction"]}')
    print(f'Input (excerpt): {sample["input"][:150]}…')
    print(f'Output: {sample["output"][:200]}…')
    print()

--- Sample 1 ---
Instruction: What are the key facts and legal issues in this case excerpt?
Input (excerpt): ned counsel has relied upon the judicial precedences in Asfaq V/s State of Rajasthan and others (2017) 15 SCC 55, Rai Sigh versus the State of Haryana…
Output: **Key facts (as per the excerpt)**  
- The case involves a discussion of several earlier judgments: *Asfaq v. State of Rajasthan and others* (2017) 15 SCC 55, *Rai Sigh v. State of Haryana* (995 SCC),…

--- Sample 2 ---
Instruction: Analyse the legal merits of the case described, citing the relevant provisions mentioned.
Input (excerpt): [Cites 34 , Cited by 2 ] Punjab-Haryana High Court Surender Alias Choti vs State Of Haryana And Others on 20 October, 2022 Author: Tejinder Singh Dhin…
Output: **Answer**

The petition challenges an order dated 28.01.2022 (Annexure P‑13) in which the application of the petitioner, Surender alias Choti, for a furlough was declined by respondent No. 6.  The Hi…

--- Sample 3 ---
Instruction

In [17]:
# Chat format (what the model sees during training)
print(json.dumps(chat_samples[0], indent=2, ensure_ascii=False))

{
  "messages": [
    {
      "role": "system",
      "content": "You are NyayaGPT, an expert Indian legal assistant trained on Indian Kanoon case law, IPC sections, constitutional provisions, and judicial judgments.\nWhen answering:\n1. Cite the relevant IPC section, article, or case law if known.\n2. Explain legal concepts in plain language.\n3. If the question is outside Indian jurisdiction or your knowledge, say so clearly.\n4. Never provide advice that could replace consultation with a licensed advocate."
    },
    {
      "role": "user",
      "content": "Summarize the court's reasoning or holding in this judgment excerpt.\n\nContext:\n[Cites 34 , Cited by 2 ] Punjab-Haryana High Court Surender Alias Choti vs State Of Haryana And Others on 20 October, 2022 Author: Tejinder Singh Dhindsa Bench: Tejinder Singh Dhindsa CRWP-1324-2022 1 IN THE HIGH COURT OF PUNJAB AND HARYANA AT CHANDIGARH CRWP-1324-2022 Date of Decision: 20.10.2022 SURENDER @ CHOTI ...Petitioner V/S STATE OF HARYAN

## Section 4 — Save Datasets (90/10 split)

In [18]:
train_path, eval_path = save_datasets(alpaca_samples, chat_samples)

# Verify
def count_lines(p):
    return sum(1 for _ in open(p, encoding='utf-8'))

print(f'\n✓ Dataset saved:')
print(f'  train.jsonl : {count_lines(train_path)} samples')
print(f'  eval.jsonl  : {count_lines(eval_path)} samples')

2026-04-19 00:58:06,933 [INFO] src.nyaya_pipeline.synthetic_generator: Saved 43 train / 5 eval samples
2026-04-19 00:58:06,934 [INFO] src.nyaya_pipeline.synthetic_generator:   train → /home/ubuntu/Fine-tuning/NyayaGPT/output/train.jsonl
2026-04-19 00:58:06,934 [INFO] src.nyaya_pipeline.synthetic_generator:   eval  → /home/ubuntu/Fine-tuning/NyayaGPT/output/eval.jsonl
2026-04-19 00:58:06,935 [INFO] src.nyaya_pipeline.synthetic_generator:   alpaca→ /home/ubuntu/Fine-tuning/NyayaGPT/output/dataset_alpaca.json

✓ Dataset saved:
  train.jsonl : 43 samples
  eval.jsonl  : 5 samples


In [ ]:
# Distribution stats
import pandas as pd

rows = []
with open(train_path, encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        asst = next(m['content'] for m in obj['messages'] if m['role'] == 'assistant')
        rows.append({'answer_len': len(asst)})

df = pd.DataFrame(rows)
print(df['answer_len'].describe().round(1))

## ✅ Day 1 Complete
**Outputs created:**
- `output/raw_judgments.json` — scraped judgments
- `output/dataset_alpaca.json` — Alpaca format
- `output/train.jsonl` — training set
- `output/eval.jsonl` — evaluation set

**Next:** Run `02_qlora_training.ipynb`